# 04 - Training data: collect, convert, validate, decontaminate, render

**Output:** `/kaggle/working/data/train.jsonl` and `val.jsonl`, where each row is `prompt` + `completion`, rendered in exactly the format BFCL's Qwen handler sends at evaluation time.

## What I verified before writing this (and what it means)
- **xLAM-60k** (`Salesforce/xlam-function-calling-60k`): gated (you must accept the terms on its page with your HF account), CC-BY-4.0, fields `query`, `tools`, `answers` (JSON). **Every example ends in a call.** The ToolACE paper trained on 25k xLAM examples and got only **11.9% irrelevance detection** (ToolACE data: 86.4%).
- **ToolACE** (`Team-ACE/ToolACE`, Apache-2.0): contains real **"no tool fits"** and **"required parameter missing, ask the user"** assistant turns. The paper's ablation: removing that type of data drops irrelevance detection to **7%**.
- => **Training on xLAM alone would likely damage the abstain behaviour BFCL measures.** We mix: xLAM for calling, ToolACE for calls + abstain + ask, and (notebook 05) our own synthetic examples for the near-miss cases.
- Reference LoRA setup from the ToolACE paper: rank 16, alpha 32, all modules, lr 1e-4, warmup 0.1, cosine, 3 epochs, batch 48. We start from the same.
- BFCL's Qwen handler builds the prompt itself and passes tool schemas **raw** (BFCL style: `"type": "dict"`, key order as in its data files). So we convert every training schema to that style and render with a copy of the handler's own `_format_prompt`. I tested the copy against the real handler on BFCL entries (identical output).


In [2]:
!pip install -q "pyarrow<24"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 5.1.0 requires pyarrow>=24.0.0, but you have pyarrow 23.0.1 which is incompatible.


In [1]:
!pip install -q -U datasets huggingface_hub scikit-learn
!pip download bfcl-eval==2026.3.23 --no-deps -q -d /kaggle/working/bfcl_whl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [3]:
import glob, json, zipfile, os, re, ast, random, collections, pathlib

whl = glob.glob("/kaggle/working/bfcl_whl/*.whl")[0]
zipfile.ZipFile(whl).extractall("/kaggle/working/bfcl_whl/src")
BFCL_DATA = "/kaggle/working/bfcl_whl/src/bfcl_eval/data"
print(sorted(os.path.basename(p) for p in glob.glob(BFCL_DATA + "/BFCL_v4_*.json"))[:6], "...")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

['BFCL_v4_format_sensitivity.json', 'BFCL_v4_irrelevance.json', 'BFCL_v4_live_irrelevance.json', 'BFCL_v4_live_multiple.json', 'BFCL_v4_live_parallel.json', 'BFCL_v4_live_parallel_multiple.json'] ...


In [4]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download

xlam = load_dataset("Salesforce/xlam-function-calling-60k", split="train")   # gated: accept terms on the dataset page first
print("xLAM rows:", len(xlam), "| columns:", xlam.column_names)

tp = hf_hub_download("Team-ACE/ToolACE", "data.json", repo_type="dataset")
raw = open(tp, encoding="utf-8").read()
try:
    toolace = json.loads(raw)
except json.JSONDecodeError:
    toolace = [json.loads(l) for l in raw.splitlines() if l.strip()]   # JSONL fallback
print("ToolACE rows:", len(toolace), "| keys:", list(toolace[0].keys()))

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

xlam_function_calling_60k.json: reconstructing file:   0%|          |  0.00B / 96.1MB            

xlam_function_calling_60k.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

xLAM rows: 60000 | columns: ['id', 'query', 'answers', 'tools']


data.json: reconstructing file:   0%|          |  0.00B / 37.2MB            

data.json: downloading bytes:           |  0.00B            

ToolACE rows: 11300 | keys: ['system', 'conversations']


In [5]:
def show(x, n=700):
    s = x if isinstance(x, str) else json.dumps(x, ensure_ascii=False)
    print(s[:n] + (" ...[cut]" if len(s) > n else ""))

print("=== xLAM[0] ===")
for k, v in xlam[0].items():
    print(f"[{k}] ({type(v).__name__})", end=" "); show(v, 500)

print("\n=== ToolACE[0] ===")
r = toolace[0]
print("[system]", end=" "); show(r["system"], 900)
for turn in r["conversations"][:4]:
    print("-", {k: (v[:300] if isinstance(v, str) else v) for k, v in turn.items()})

=== xLAM[0] ===
[id] (int) 0
[query] (str) Where can I find live giveaways for beta access and games?
[answers] (str) [{"name": "live_giveaways_by_type", "arguments": {"type": "beta"}}, {"name": "live_giveaways_by_type", "arguments": {"type": "game"}}]
[tools] (str) [{"name": "live_giveaways_by_type", "description": "Retrieve live giveaways from the GamerPower API based on the specified type.", "parameters": {"type": {"description": "The type of giveaways to retrieve (e.g., game, loot, beta).", "type": "str", "default": "game"}}}]

=== ToolACE[0] ===
[system] You are an expert in composing functions. You are given a question and a set of possible functions. 
Based on the question, you will need to make one or more function/tool calls to achieve the purpose. 
If none of the function can be used, point it out. If the given question lacks the parameters required by the function,
also point it out. You should only return the function call in tools call sections.
Here is a list of functions

In [7]:
# ---------- schema normalisation (xLAM python-style types -> BFCL style) ----------
TYPE_MAP = {"str": "string", "string": "string", "int": "integer", "integer": "integer", "float": "float",
            "number": "float", "double": "float", "bool": "boolean", "boolean": "boolean",
            "list": "array", "array": "array", "tuple": "array", "dict": "dict", "object": "dict"}

def norm_type(t):
    t = str(t).lower().strip()
    optional = "optional" in t
    base = re.sub(r"\[.*", "", t.split(",")[0]).strip()
    return TYPE_MAP.get(base, "string"), optional

def maybe_json(x):
    if isinstance(x, str):
        try: return json.loads(x)
        except json.JSONDecodeError: return ast.literal_eval(x)
    return x

def xlam_tool_to_bfcl(t):
    props, required = {}, []
    for pname, p in (t.get("parameters") or {}).items():
        if not isinstance(p, dict): p = {"type": "string", "description": str(p)}
        typ, opt = norm_type(p.get("type", "string"))
        d = {"type": typ, "description": p.get("description", "")}
        if typ == "array": d["items"] = {"type": "string"}
        props[pname] = d
        if not (opt or "default" in p): required.append(pname)
    return {"name": t["name"], "description": t.get("description", ""),
            "parameters": {"type": "dict", "properties": props, "required": required}}

# ---------- ToolACE bracket-syntax parser: "[f(a=1, b='x'), g(c=[1,2])]" ----------
def _parse_kwargs(argstr):
    try:
        node = ast.parse(f"f({argstr})", mode="eval").body
        if node.args: return None                      # positional args: cannot map to names
        return {kw.arg: ast.literal_eval(kw.value) for kw in node.keywords}
    except Exception:
        return None

def parse_bracket_calls(text, names):
    t = text.strip()
    if not (t.startswith("[") and t.endswith("]")): return None
    inner, i, calls = t[1:-1], 0, []
    by_len = sorted(names, key=len, reverse=True)       # names may contain spaces / dots
    while i < len(inner):
        while i < len(inner) and inner[i] in " ,\n": i += 1
        if i >= len(inner): break
        name = next((n for n in by_len if inner.startswith(n + "(", i)), None)
        if name is None: return None
        j = k = i + len(name) + 1
        depth, quote = 1, None
        while k < len(inner) and depth > 0:
            ch = inner[k]
            if quote:
                if ch == "\\": k += 2; continue
                if ch == quote: quote = None
            elif ch in "\"'": quote = ch
            elif ch in "([{": depth += 1
            elif ch in ")]}": depth -= 1
            k += 1
        if depth != 0: return None
        args = _parse_kwargs(inner[j:k - 1])
        if args is None: return None
        calls.append({"name": name, "arguments": args})
        i = k
    return calls or None

def parse_tool_list(s):
    s = s.strip()
    if "]" in s: s = s[: s.rfind("]") + 1]
    for loader in (json.loads, ast.literal_eval):
        try:
            v = loader(s)
            if isinstance(v, dict): v = [v]
            if isinstance(v, list) and v and all(isinstance(x, dict) and "name" in x for x in v): return v
        except Exception:
            pass
    return None

def coerce_args(calls, tools):
    """13.0 -> 13 where the schema says integer."""
    by = {t["name"]: t["parameters"].get("properties", {}) for t in tools}
    for c in calls:
        for k, v in list(c["arguments"].items()):
            typ = by.get(c["name"], {}).get(k, {}).get("type")
            if typ == "integer" and isinstance(v, float) and v.is_integer():
                c["arguments"][k] = int(v)
    return calls

ASK_RE = re.compile(r"(lacks?|missing|requires?|needs?|please (provide|specify|share|tell)|could you (please )?(provide|specify|share)|"
                    r"can you (please )?(provide|specify|share)|provide the)", re.I)

# ---------- record -> internal example ----------
def xlam_to_example(rec, i):
    try:
        tools = [xlam_tool_to_bfcl(t) for t in maybe_json(rec["tools"])]
        answers = maybe_json(rec["answers"])
    except Exception as e:
        return None, "xlam_parse_error"
    if not tools or not answers: return None, "xlam_empty"
    calls = [{"name": a["name"], "arguments": a.get("arguments", {})} for a in answers]
    return {"id": f"xlam-{rec.get('id', i)}", "source": "xlam", "query": rec["query"], "tools": tools,
            "target": {"type": "call", "calls": coerce_args(calls, tools)}}, None

def toolace_to_example(rec, i):
    marker = "Here is a list of functions in JSON format that you can invoke."
    system = rec.get("system", "")
    if marker not in system: return None, "toolace_no_marker"
    tools = parse_tool_list(system.split(marker, 1)[1])
    if not tools: return None, "toolace_tool_parse"
    convs = rec.get("conversations") or []
    if len(convs) < 2: return None, "toolace_short"
    g = lambda t, a, b: t.get(a) if t.get(a) is not None else t.get(b)
    r0, v0, r1, v1 = g(convs[0], "from", "role"), g(convs[0], "value", "content"), g(convs[1], "from", "role"), g(convs[1], "value", "content")
    if r0 != "user" or r1 != "assistant" or not isinstance(v0, str) or not isinstance(v1, str):
        return None, "toolace_unexpected_roles"
    names = [t["name"] for t in tools]
    t1 = v1.strip()
    if t1.startswith("[") and t1.endswith("]"):
        calls = parse_bracket_calls(t1, names)
        if calls is None: return None, "toolace_call_parse"
        target = {"type": "call", "calls": coerce_args(calls, tools)}
    else:
        target = {"type": "text", "kind": "ask" if ASK_RE.search(t1) else "abstain_or_other", "text": t1}
    return {"id": f"toolace-{i}", "source": "toolace", "query": v0, "tools": tools, "target": target}, None

In [8]:
names = ["performanceArt.get_upcoming_events", "Get Live Events Count by Sport", "Get All NBA Stats", "Get Latest Team Media"]

c = parse_bracket_calls('[performanceArt.get_upcoming_events(start_date="2021-04-01", end_date="2021-05-01", category="Theatre"), '
                        'performanceArt.get_upcoming_events(start_date="2021-04-01", end_date="2021-05-01", category="Dance")]', names)
assert len(c) == 2 and c[1]["arguments"]["category"] == "Dance"

c = parse_bracket_calls('[Get Live Events Count by Sport(sport="soccer", locale="en_US", timezone=-4.0), '
                        'Get All NBA Stats(seasons=["2025"], page=1, per_page=10), Get Latest Team Media(team_id=13.0)]', names)
assert [x["name"] for x in c] == ["Get Live Events Count by Sport", "Get All NBA Stats", "Get Latest Team Media"]
assert c[0]["arguments"]["timezone"] == -4.0 and c[1]["arguments"]["seasons"] == ["2025"]

c = parse_bracket_calls('[Get All NBA Stats(seasons=["a, b)"], page=1)]', names)       # brackets/commas inside a string
assert c[0]["arguments"]["seasons"] == ["a, b)"]

assert parse_bracket_calls("The provided functions do not support that.", names) is None      # plain text
assert parse_bracket_calls('[Unknown.fn(a=1)]', names) is None                                # unknown function
assert parse_bracket_calls('[Get All NBA Stats(1, 2)]', names) is None                        # positional args
print("parser tests passed")

parser tests passed


In [9]:
examples, drops = [], collections.Counter()
for i, rec in enumerate(xlam):
    ex, why = xlam_to_example(rec, i)
    (examples.append(ex) if ex else drops.update([why]))
for i, rec in enumerate(toolace):
    ex, why = toolace_to_example(rec, i)
    (examples.append(ex) if ex else drops.update([why]))

print("converted:", collections.Counter((e["source"], e["target"]["type"] + (":" + e["target"].get("kind", "") if e["target"]["type"] == "text" else "")) for e in examples))
print("dropped at conversion:", dict(drops))

converted: Counter({('xlam', 'call'): 60000})
dropped at conversion: {'toolace_no_marker': 11300}


In [10]:
DROP_UNGROUNDED = True

def norm_txt(s): return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]+", " ", str(s).lower())).strip()

def validate(ex):
    if ex["target"]["type"] != "call": return []
    by = {t["name"]: t for t in ex["tools"]}
    problems = []
    for c in ex["target"]["calls"]:
        t = by.get(c["name"])
        if t is None: problems.append("unknown_function"); continue
        params = t["parameters"]; props = params.get("properties", {}); req = params.get("required", [])
        if any(r not in c["arguments"] for r in req): problems.append("missing_required")
        if any(k not in props for k in c["arguments"]): problems.append("unknown_argument")
        q = norm_txt(ex["query"])
        for k, v in c["arguments"].items():
            p = props.get(k, {})
            if k in req and isinstance(v, str) and len(v) >= 3 and "enum" not in p and norm_txt(v) not in q:
                problems.append("ungrounded_required_string")
    return sorted(set(problems))

val_counts, valid, flagged = collections.Counter(), [], []
for ex in examples:
    probs = validate(ex)
    val_counts.update(probs or ["ok"])
    hard = [p for p in probs if p != "ungrounded_required_string"]
    if hard or (DROP_UNGROUNDED and "ungrounded_required_string" in probs):
        flagged.append((ex, probs)); continue
    valid.append(ex)
print("validation flags:", dict(val_counts))
print(f"kept {len(valid)} / {len(examples)}")

print("\nA few dropped examples (read these):")
for ex, probs in random.Random(1).sample(flagged, min(5, len(flagged))):
    c = ex["target"]["calls"][0]
    print(f"- {probs} | Q: {ex['query'][:120]!r} | call: {c['name']}({json.dumps(c['arguments'])[:140]})")

validation flags: {'ok': 59023, 'missing_required': 275, 'ungrounded_required_string': 709}
kept 59023 / 60000

A few dropped examples (read these):
- ['ungrounded_required_string'] | Q: 'How old is someone born on January 1, 1990?' | call: calculate_age({"birthdate": "1990-01-01"})
- ['ungrounded_required_string'] | Q: 'Can I check-in to the Queens Hotel in Berlin on March 15 and check-out on March 20?' | call: is_hotel_available({"hotel": "Queens Hotel", "city": "Berlin, Germany", "checkin": "2022-03-15", "checkout": "2022-03-20"})
- ['ungrounded_required_string'] | Q: 'Is the Queens Hotel in Berlin available for check-in on March 12, 2022 and check-out on March 15, 2022?' | call: is_hotel_available({"hotel": "Queens Hotel", "city": "Berlin, Germany", "checkin": "2022-03-12", "checkout": "2022-03-15"})
- ['ungrounded_required_string'] | Q: 'Check if there is a majority element in the list [1, 1, 2, 2, 3, 3, 4, 4] and generate a random string of length 10 with' | call: find_majority_e

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

THRESH = 0.80
EXTRA_BLOCK_QUERIES = []        # e.g. paste the 45 probe queries from notebook 02, and your future held-out test queries

def collect_user_texts(obj, out):
    if isinstance(obj, dict):
        if obj.get("role") == "user" and isinstance(obj.get("content"), str): out.append(obj["content"])
        for v in obj.values(): collect_user_texts(v, out)
    elif isinstance(obj, list):
        for v in obj: collect_user_texts(v, out)

bfcl_q = []
for path in glob.glob(BFCL_DATA + "/BFCL_v4_*.json"):
    for line in open(path, encoding="utf-8"):
        if not line.strip(): continue
        try: rec = json.loads(line)
        except json.JSONDecodeError: break          # not a JSONL file (e.g. BFCL_v4_format_sensitivity.json is an id list)
        if isinstance(rec, dict): collect_user_texts(rec.get("question", []), bfcl_q)
bfcl_q = sorted(set(q for q in bfcl_q + EXTRA_BLOCK_QUERIES if q.strip()))
print("BFCL + extra blocked queries:", len(bfcl_q))

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True).fit(bfcl_q + [e["query"] for e in valid])
B = vec.transform(bfcl_q)
exact = set(norm_txt(q) for q in bfcl_q)

clean, removed_exact, removed_near, max_sims = [], 0, 0, []
for s in range(0, len(valid), 4000):
    chunk = valid[s:s + 4000]
    sims = (vec.transform([e["query"] for e in chunk]) @ B.T).max(axis=1).toarray().ravel()
    for e, sim in zip(chunk, sims):
        max_sims.append(sim)
        if norm_txt(e["query"]) in exact: removed_exact += 1
        elif sim >= THRESH: removed_near += 1
        else: clean.append(e)
print(f"removed exact: {removed_exact} | removed near (>= {THRESH}): {removed_near} | remaining: {len(clean)}")
import numpy as np
print("similarity percentiles (50/90/99/max):", [round(float(np.percentile(max_sims, p)), 2) for p in (50, 90, 99, 100)])

BFCL + extra blocked queries: 4826
removed exact: 6 | removed near (>= 0.8): 193 | remaining: 58824
similarity percentiles (50/90/99/max): [0.14, 0.3, 0.64, 1.0]


In [12]:
THINK_OFF = True          # True: Qwen/Qwen3-1.7B (hybrid)   False: Qwen3-4B-Instruct-2507
THINK_OFF_BLOCK = "<think>\n\n</think>\n\n"

def format_prompt_bfcl(messages, function):
    """Copy of QwenFCHandler._format_prompt for user-only / first-turn histories."""
    p = ""
    if len(function) > 0:
        p += "<|im_start|>system\n"
        if messages[0]["role"] == "system": p += messages[0]["content"] + "\n\n"
        p += "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>"
        for tool in function: p += f"\n{json.dumps(tool)}"
        p += '\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{"name": <function-name>, "arguments": <args-json-object>}\n</tool_call><|im_end|>\n'
    for m in messages:
        if m["role"] == "user": p += f"<|im_start|>user\n{m['content']}<|im_end|>\n"
    return p + "<|im_start|>assistant\n"

def tool_call_block(c):
    return '<tool_call>\n{"name": "' + c["name"] + '", "arguments": ' + json.dumps(c["arguments"]) + "}\n</tool_call>"

def render(ex):
    prompt = format_prompt_bfcl([{"role": "user", "content": ex["query"]}], ex["tools"])
    if THINK_OFF: prompt += THINK_OFF_BLOCK
    t = ex["target"]
    body = "\n".join(tool_call_block(c) for c in t["calls"]) if t["type"] == "call" else t["text"]
    return prompt, body + "<|im_end|>"

p, c_ = render(next(e for e in clean if e["target"]["type"] == "call"))
print(p[-700:]); print("---- COMPLETION ----"); print(c_)

gnatures within <tools></tools> XML tags:
<tools>
{"name": "live_giveaways_by_type", "description": "Retrieve live giveaways from the GamerPower API based on the specified type.", "parameters": {"type": "dict", "properties": {"type": {"type": "string", "description": "The type of giveaways to retrieve (e.g., game, loot, beta)."}}, "required": []}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Where can I find live giveaways for beta access and games?<|im_end|>
<|im_start|>assistant
<think>

</think>


---- COMPLETION ----
<tool_call>
{"name": "live_giveaways_by_type", "arguments": {"type": "beta"}}
</tool_call>
<tool_call>
{"name": "live_giveaways_by_type", "arguments": {"type": "game"}}
</tool_call><|im_end|>


In [13]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

N_XLAM = 4000
MAX_LEN = 2048
rng = random.Random(0)

xl  = [e for e in clean if e["source"] == "xlam"]
ta  = [e for e in clean if e["source"] == "toolace"]
mix = rng.sample(xl, min(N_XLAM, len(xl))) + ta
rng.shuffle(mix)

rows, too_long = [], 0
for e in mix:
    prompt, completion = render(e)
    n = len(tok(prompt + completion)["input_ids"])
    if n > MAX_LEN: too_long += 1; continue
    t = e["target"]
    rows.append({"id": e["id"], "source": e["source"], "kind": t["type"] if t["type"] == "call" else t["kind"],
                 "prompt": prompt, "completion": completion, "n_tokens": n})
print("dropped for length >", MAX_LEN, ":", too_long)

n_val = int(0.05 * len(rows))
val, train = rows[:n_val], rows[n_val:]
os.makedirs("/kaggle/working/data", exist_ok=True)
for name, part in (("train", train), ("val", val)):
    with open(f"/kaggle/working/data/{name}.jsonl", "w") as f:
        for r in part: f.write(json.dumps(r) + "\n")

import pandas as pd
df = pd.DataFrame(train)
print(f"train {len(train)} | val {len(val)}")
print(df.groupby(["source", "kind"]).size().unstack(fill_value=0))
print("\nshare of non-call targets in train: %.1f%%" % (100 * (df.kind != "call").mean()))
print("token length percentiles (50/90/99):", [int(np.percentile(df.n_tokens, p)) for p in (50, 90, 99)])

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

dropped for length > 2048 : 0
train 3800 | val 200
kind    call
source      
xlam    3800

share of non-call targets in train: 0.0%
token length percentiles (50/90/99): [449, 758, 1076]


In [14]:
for kind in ["call", "ask", "abstain_or_other"]:
    sub = [r for r in train if r["kind"] == kind]
    print("#" * 90); print(kind, "| n =", len(sub))
    for r in random.Random(3).sample(sub, min(3, len(sub))):
        user = r["prompt"].split("<|im_start|>user\n")[-1].split("<|im_end|>")[0]
        tools = re.findall(r'\{"name": "([^"]+)"', r["prompt"].split("<tools>")[1].split("</tools>")[0])
        print(f"\n[{r['id']}] tools: {tools}")
        print("USER:", user[:300])
        print("TARGET:", r["completion"][:400])

##########################################################################################
call | n = 3800

[xlam-7735] tools: []
USER: Retrieve detailed lead data for campaign ID 'C1002' from the Fomo API.
TARGET: <tool_call>
{"name": "get_campaign_lead_from_id", "arguments": {"is_id": "C1002"}}
</tool_call><|im_end|>

[xlam-32038] tools: []
USER: Retrieve 20 reviews for the app 'com.fitness.app' from Canada, sorted by rating, and check the deliverability of 'fitnesspro@example.com'.
TARGET: <tool_call>
{"name": "app_reviews", "arguments": {"app_id": "com.fitness.app", "region": "ca", "sort_by": "RATING", "limit": 20}}
</tool_call>
<tool_call>
{"name": "emailverifier", "arguments": {"email": "fitnesspro@example.com"}}
</tool_call><|im_end|>

[xlam-42630] tools: []
USER: What are the albums released by Post Malone, and what is his discography overview?
TARGET: <tool_call>
{"name": "get_artist_s_albums", "arguments": {"spotify_url": "https://open.spotify.com/artist/246dkjvS1zLTtiykXe5h6